# SFF Build Checker
Ez a notebook egy intelligens, adatközpontú (Data-driven) ágenst valósít meg LangGraph alapokon, amely Small Form Factor (SFF) számítógépek tervezésében segít.

A rendszer legfőbb architekturális sajátossága a **Szigorú Zéró-Tudás (Zero-Knowledge) megközelítés**: az ágens nem használhatja a saját nyelvi modelljébe kódolt (hallucinációra hajlamos) hardveres ismereteit. Minden adatot valós időben, külső API-kon keresztül kell beszereznie, validálnia és a lokális memóriájában (State) tárolnia.

**Implementált Toolok (Eszköztár):**
*   **Web Scraper (V2):** Tavily alapú URL keresés + ScraperAPI (Prémium proxy a Cloudflare védelem megkerülésére) + Rugalmas Regex adatkinyerés.
*   **Valutaváltó:** Valós idejű konverzió az ExchangeRate-API segítségével.
*   **PSU Kalkulátor:** Mérnöki számítás alapú tápegység-kalkulátor (TDP + Alaprendszer + 20% tartalék).
*   **State Management:** Helyi adatbázis (`current_pc_build`) menedzselése (hozzáadás, törlés, ürítés).
*   **Fórum Sentiment:** Reddit PRAW/Tavily keresés kombinálva a Hugging Face Inference API-val (RoBERTa alapú NLP modell) a közösségi megbízhatóság mérésére.

# Enviroment variables
API kulcsok: A külső szolgáltatások (Tavily, ScraperAPI) eléréséhez szükséges azonosítók tárolása.

In [ ]:
import os
SCRAPER_API_KEY = ""
os.environ["TAVILY_API_KEY"] = ""
os.environ["GOOGLE_API_KEY"] = ""
os.environ["HF_TOKEN"] = ""

#Importok és setup
Library-k: Szükséges csomagok (BS4, Requests, LangChain/Tavily) behúzása az adatgyűjtéshez.  

Regex: Reguláris kifejezések a szöveges adatok (mm, Watt) kinyeréséhez.

In [ ]:
!pip install -U langchain-tavily beautifulsoup4 requests langchain-google-genai langgraph

In [ ]:
from dataclasses import dataclass
from typing import Optional, Dict
import time
import requests
import re
from bs4 import BeautifulSoup
from langchain_tavily import TavilySearch
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import convert_to_messages
from huggingface_hub import InferenceClient

#Globális tárolók/változók
PCComponent: Dataclass az alkatrészek egységes tárolásához (típus, név, TDP, dimenziók).  

current_pc_build: Lokális szótár (dict), ami a gép aktuális állapotát tárolja (típusonként max 1 alkatrész).

In [ ]:
'''
#ELAVULT - Web scraper V1-hez írt adatstruktúra
@dataclass
class PCComponent:
    """Egy PC alkatrész adatmodellje."""
    type: str      # pl. "CPU", "GPU", "Motherboard"
    name: str      # pl. "Ryzen 7 9800X3D"
    tdp: Optional[int] = None      # Fogyasztás Wattban (opcionális)
    dimensions: Optional[dict] = None # Méretek (opcionális)


'''

'\n#ELAVULT - Web scraper V1-hez írt adatstruktúra\n@dataclass\nclass PCComponent:\n    """Egy PC alkatrész adatmodellje."""\n    type: str      # pl. "CPU", "GPU", "Motherboard"\n    name: str      # pl. "Ryzen 7 9800X3D"\n    tdp: Optional[int] = None      # Fogyasztás Wattban (opcionális)\n    dimensions: Optional[dict] = None # Méretek (opcionális)\n\n\n'

In [ ]:
#ÚJ - Web scraper V2-höz írt új adatstruktúra
@dataclass
class PCComponent:
    """Egy univerzális PC alkatrész adatmodellje."""
    name: str
    type: str  # Pl.: 'GPU', 'CPU', 'Motherboard', 'PSU', 'RAM'

    # Közös tulajdonságok (Mindenkinek lehet)
    tdp: Optional[int] = None
    price_huf: Optional[int] = None

    # GPU Specifikus méretek
    length_mm: Optional[float] = None
    width_mm: Optional[float] = None
    height_mm: Optional[float] = None

    # CPU / Alaplap specifikus
    socket: Optional[str] = None

    # PSU (Tápegység) specifikus
    wattage: Optional[int] = None
    form_factor: Optional[str] = None # Pl.: 'SFX', 'ATX'

current_pc_build: Dict[str, PCComponent] = {}

# Web scraper V1
* Search tool, hogy ne kelljen tippelni az URL-t (visszaadja az első találatot `site:pcpartpicker.com <alkatrész neve>`): `from langchain_tavily import TavilySearch`
* Scraper:
    * [ScraperAPI](https://www.scraperapi.com)  (Limitált ingyenes használat)
    * Regex a dimenziók megkereséséhez (GPU)
    * Regex a TDP kereséséhez
  



##ELAVULT
Mást használok mostmár, de bent tartom ha egyszer szükséges lenne fallback toolokként

## Videó kártya gyártók
Keresési szótár: Gyártó-specifikus keresési módosítók, hogy ne fórumokat, hanem hivatalos tech spec oldalakat találjunk meg.

In [ ]:
'''
MANUFACTURER_QUERIES = {
    # --- Nagy AIB Partnerek (Több chipgyártóval is dolgoznak) ---
    "asus": "site:asus.com inurl:techspec",
    "msi": "site:msi.com inurl:Specification",
    "gigabyte": "site:gigabyte.com inurl:Specification",
    "asrock": "site:asrock.com specification",

    # --- Exkluzív AMD Partnerek ---
    "sapphire": "site:sapphiretech.com specifications",
    "powercolor": "site:powercolor.com specification",
    "xfx": "site:xfxforce.com specifications",

    # --- Exkluzív NVIDIA Partnerek ---
    "zotac": "site:zotac.com spec",
    "pny": "site:pny.com specifications",        #problémás mivel pdf-et ad
    "palit": "site:palit.com specification",
    "gainward": "site:gainward.com specification",
    "inno3d": "site:inno3d.com specification",
    "kfa2": "site:kfa2.com specification",       # Európai piacon gyakori (Galax almárka)
    "galax": "site:galax.com specification",
    "colorful": "site:en.colorful.cn specification",

    # --- Intel Arc partnerek ---
    "sparkle": "site:sparkle.com.tw specifications",
    "acer": "site:acer.com tech specs",          # Predator BiFrost kártyákhoz

    # --- Referencia / Saját gyártású kártyák ---
    "founders": "site:nvidia.com specs",         # NVIDIA Founders Edition
    "intel": "site:intel.com specifications",    # Intel Arc Limited Edition

    # --- Az "amd" és "nvidia" kulcsszavakat a lista végére tesszük,
    # hogy az algoritmus először az AIB partnereket találja meg!
    "amd": "site:amd.com specifications",        # AMD Referencia kártyák
    "nvidia": "site:nvidia.com specs"            # Biztonsági háló, ha a Founders szót kihagyják
}
'''

'\nMANUFACTURER_QUERIES = {\n    # --- Nagy AIB Partnerek (Több chipgyártóval is dolgoznak) ---\n    "asus": "site:asus.com inurl:techspec",\n    "msi": "site:msi.com inurl:Specification",\n    "gigabyte": "site:gigabyte.com inurl:Specification",\n    "asrock": "site:asrock.com specification",\n\n    # --- Exkluzív AMD Partnerek ---\n    "sapphire": "site:sapphiretech.com specifications",\n    "powercolor": "site:powercolor.com specification",\n    "xfx": "site:xfxforce.com specifications",\n\n    # --- Exkluzív NVIDIA Partnerek ---\n    "zotac": "site:zotac.com spec",\n    "pny": "site:pny.com specifications",        #problémás mivel pdf-et ad\n    "palit": "site:palit.com specification", \n    "gainward": "site:gainward.com specification",\n    "inno3d": "site:inno3d.com specification",\n    "kfa2": "site:kfa2.com specification",       # Európai piacon gyakori (Galax almárka)\n    "galax": "site:galax.com specification",\n    "colorful": "site:en.colorful.cn specification",\n\n    # 

## A kereső és scraper

###Méret/dimenzió
`get_gpu_dimensions`: Gyártói oldalakat céloz meg a pontos L x W x H méretekért.

In [ ]:
'''
def get_gpu_dimensions(component_name: str) -> dict:
    """
    Kikeresi a hivatalos gyártói oldalt egy hardverhez, és lekaparja a fizikai méreteit (Hossz x Szélesség x Magasság).
    Csak a méretek (dimensions) megkeresésére használd!
    """
    result = {"component_name": component_name, "url": None, "dimensions": None, "status": "Folyamatban"}

    # 1. Gyártó azonosítása
    component_lower = component_name.lower()
    search_modifier = ""
    for manufacturer, query in MANUFACTURER_QUERIES.items():
        if manufacturer in component_lower:
            search_modifier = query
            break

    if not search_modifier:
        result["status"] = "Hiba: Ismeretlen gyártó."
        return result

    # 2. Tavily Keresés
    try:
        web_search = TavilySearch(max_results=3)
        search_results = web_search.invoke(f"{component_name} {search_modifier}")

        target_url = None
        if search_results and "results" in search_results:
            for res in search_results["results"]:
                target_url = res["url"]
                break

        if not target_url:
            result["status"] = "Hiba: Nem találtam specifikációs oldalt."
            return result
        result["url"] = target_url

    except Exception as e:
        result["status"] = f"Tavily hiba: {str(e)}"
        return result

    # 3. ScraperAPI Letöltés
    try:
        payload = {'api_key': SCRAPER_API_KEY, 'url': target_url, 'render': 'false'}
        response = requests.get('https://api.scraperapi.com/', params=payload, timeout=30)

        if response.status_code != 200:
            result["status"] = f"HTTP Hiba: {response.status_code}"
            return result

        soup = BeautifulSoup(response.text, 'html.parser')
        for element in soup(["script", "style", "nav", "footer", "header"]):
            element.extract()
        text = soup.get_text(separator=' ', strip=True)

        # Méret Regex
        dim_pattern_1 = r'(\d+(?:[.,]\d+)?)\s*(?:mm)?\s*[xX*]\s*(\d+(?:[.,]\d+)?)\s*(?:mm)?\s*[xX*]\s*(\d+(?:[.,]\d+)?)\s*mm'
        dim_pattern_2 = r'[L]*\s*=?\s*(\d+(?:[.,]\d+)?)\s*(?:mm)?\s*.*?[W]*\s*=?\s*(\d+(?:[.,]\d+)?)\s*(?:mm)?\s*.*?[H]*\s*=?\s*(\d+(?:[.,]\d+)?)\s*mm'

        dim_matches = re.findall(dim_pattern_1, text)
        if not dim_matches:
            dim_matches = re.findall(dim_pattern_2, text, re.IGNORECASE)

        if dim_matches:
            l, w, h = dim_matches[0]
            result["dimensions"] = {"length_mm": float(l.replace(',', '.')),
                                    "width_mm": float(w.replace(',', '.')),
                                    "height_mm": float(h.replace(',', '.'))}
            result["status"] = "Sikeres"
        else:
            result["status"] = "Nem találtam méretadatot."

    except Exception as e:
        result["status"] = f"Scraper hiba: {str(e)}"

    return result
'''

<>:56: SyntaxWarning: invalid escape sequence '\d'
<>:56: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_11939/1100243091.py:56: SyntaxWarning: invalid escape sequence '\d'
  dim_pattern_1 = r'(\d+(?:[.,]\d+)?)\s*(?:mm)?\s*[xX*]\s*(\d+(?:[.,]\d+)?)\s*(?:mm)?\s*[xX*]\s*(\d+(?:[.,]\d+)?)\s*mm'


'\ndef get_gpu_dimensions(component_name: str) -> dict:\n    """\n    Kikeresi a hivatalos gyártói oldalt egy hardverhez, és lekaparja a fizikai méreteit (Hossz x Szélesség x Magasság).\n    Csak a méretek (dimensions) megkeresésére használd!\n    """\n    result = {"component_name": component_name, "url": None, "dimensions": None, "status": "Folyamatban"}\n\n    # 1. Gyártó azonosítása\n    component_lower = component_name.lower()\n    search_modifier = ""\n    for manufacturer, query in MANUFACTURER_QUERIES.items():\n        if manufacturer in component_lower:\n            search_modifier = query\n            break\n\n    if not search_modifier:\n        result["status"] = "Hiba: Ismeretlen gyártó."\n        return result\n\n    # 2. Tavily Keresés\n    try:\n        web_search = TavilySearch(max_results=3)\n        search_results = web_search.invoke(f"{component_name} {search_modifier}")\n\n        target_url = None\n        if search_results and "results" in search_results:\n      

###TDP
`get_component_tdp`: Célzottan a PCPartPicker-ről húzza le a tiszta fogyasztási adatokat.

In [ ]:
'''
def get_component_tdp(component_name: str) -> dict:
    """
    Rákeres a megadott számítógép alkatrészre a PCPartPicker oldalán, és lekaparja a TDP (Thermal Design Power) értékét wattban.
    Kizárólag a fogyasztási adatok (TDP) kiderítésére használd!
    """
    result = {"component_name": component_name, "url": None, "tdp": None, "status": "Folyamatban"}

    # 1. Tavily Keresés (Célzottan PCPartPicker termékoldalra)
    try:
        web_search = TavilySearch(max_results=3)
        search_query = f"site:pcpartpicker.com/product {component_name}"
        search_results = web_search.invoke(search_query)

        target_url = None
        if search_results and "results" in search_results:
            for res in search_results["results"]:
                url_lower = res["url"].lower()
                # Biztosítjuk, hogy a /product/ mappában legyen, de kizárjuk az aloldalakat
                if "pcpartpicker.com/product/" in url_lower and "/reviews" not in url_lower and "/history" not in url_lower:
                    target_url = res["url"]
                    break

        if not target_url:
            result["status"] = "Hiba: Nem találtam PCPartPicker linket."
            return result
        result["url"] = target_url

    except Exception as e:
        result["status"] = f"Tavily hiba: {str(e)}"
        return result

    # 2. ScraperAPI Letöltés
    try:
        payload = {'api_key': SCRAPER_API_KEY, 'url': target_url, 'render': 'false'}
        response = requests.get('https://api.scraperapi.com/', params=payload, timeout=30)

        if response.status_code != 200:
            result["status"] = f"HTTP Hiba: {response.status_code}"
            return result

        soup = BeautifulSoup(response.text, 'html.parser')
        text = soup.get_text(separator=' ', strip=True)

        # TDP Regex (A PCPartPicker általában egyértelműen "TDP" formában listázza)
        tdp_pattern = r'(?:TDP|Thermal Design Power)[\s\S]{0,50}?(\d+)\s*W'
        tdp_matches = re.findall(tdp_pattern, text, re.IGNORECASE)

        if tdp_matches:
            result["tdp"] = {"value": int(tdp_matches[0]), "unit": "W"}
            result["status"] = "Sikeres"
        else:
            result["status"] = "Nem találtam TDP adatot az oldalon."

    except Exception as e:
        result["status"] = f"Scraper hiba: {str(e)}"

    return result
'''

<>:46: SyntaxWarning: invalid escape sequence '\s'
<>:46: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_11939/327216912.py:46: SyntaxWarning: invalid escape sequence '\s'
  tdp_pattern = r'(?:TDP|Thermal Design Power)[\s\S]{0,50}?(\d+)\s*W'


'\ndef get_component_tdp(component_name: str) -> dict:\n    """\n    Rákeres a megadott számítógép alkatrészre a PCPartPicker oldalán, és lekaparja a TDP (Thermal Design Power) értékét wattban.\n    Kizárólag a fogyasztási adatok (TDP) kiderítésére használd!\n    """\n    result = {"component_name": component_name, "url": None, "tdp": None, "status": "Folyamatban"}\n\n    # 1. Tavily Keresés (Célzottan PCPartPicker termékoldalra)\n    try:\n        web_search = TavilySearch(max_results=3)\n        search_query = f"site:pcpartpicker.com/product {component_name}"\n        search_results = web_search.invoke(search_query)\n\n        target_url = None\n        if search_results and "results" in search_results:\n            for res in search_results["results"]:\n                url_lower = res["url"].lower()\n                # Biztosítjuk, hogy a /product/ mappában legyen, de kizárjuk az aloldalakat\n                if "pcpartpicker.com/product/" in url_lower and "/reviews" not in url_lower 

#Web scraper V2
## Miért volt szükség a V2 Scraperre?
A V1-es verzió gyakran ütközött a TechPowerUp Cloudflare bot-védelmébe (CAPTCHA), ami miatt a tiszta HTML helyett egy hibaoldalt kapott a rendszer, így a Regex nem talált adatot ("Silent Failure").
A V2-es verzió ezt a problémát hárítja el:
1. **Prémium Proxy Routing:** A ScraperAPI `premium=true` paraméterével a kérések lakossági (residential) IP-kről érkeznek, átütve a bot-checket.
2. **Robusztus Regex:** A kód immár tolerálja a változó szóközöket, tizedesvessző/pont cseréket és a felesleges HTML tagek (pl. áthúzott árak `<s>`) előzetes szűrését.

##GPU scraper

In [ ]:
def get_GPU_specs(component_name: str, max_retries: int = 3) -> dict:
    """Kikeresi az asztali videókártya adatait a TechPowerUp adatbázisából, beépített újrapróbálkozással."""

    for attempt in range(max_retries):
        result = {"component_name": component_name, "url": None, "dimensions": None, "tdp": None, "status": "Folyamatban"}

        try:
            # 1. Keresési kifejezés
            search_query = f"{component_name} site:techpowerup.com/gpu-specs -mobile -laptop -max-q"
            web_search = TavilySearch(max_results=5)
            search_results = web_search.invoke(search_query)

            # 2. URL szűrő
            def is_valid_desktop_url(url):
                url_lower = url.lower()
                if "techpowerup.com/gpu-specs/" not in url_lower:
                    return False
                if any(keyword in url_lower for keyword in ["-mobile", "-laptop", "-max-q"]):
                    return False
                return True

            target_url = next((res["url"] for res in search_results.get("results", []) if is_valid_desktop_url(res["url"])), None)

            if not target_url:
                result["status"] = "Hiba: Nem találtam megfelelő asztali videókártya adatlapot."
                return result

            result["url"] = target_url

            # 3. ScraperAPI hívás
            payload = {'api_key': SCRAPER_API_KEY, 'url': target_url, 'render': 'false', 'premium': 'true'}
            response = requests.get('https://api.scraperapi.com/', params=payload, timeout=60)

            if response.status_code != 200:
                result["status"] = f"HTTP Hiba: {response.status_code}"
            else:
                soup = BeautifulSoup(response.text, 'html.parser')
                for s_tag in soup.find_all('s'): s_tag.decompose()
                for element in soup(["script", "style", "nav", "footer", "header"]): element.decompose()
                text = soup.get_text(separator=' ', strip=True)

                length_match = re.search(r'Length\D{0,30}?([0-9]+(?:[.,][0-9]+)?)\s*mm', text, re.IGNORECASE)
                width_match = re.search(r'(?<!Slot\s)Width\D{0,30}?([0-9]+(?:[.,][0-9]+)?)\s*mm', text, re.IGNORECASE)
                height_match = re.search(r'Height\D{0,30}?([0-9]+(?:[.,][0-9]+)?)\s*mm', text, re.IGNORECASE)
                tdp_match = re.search(r'TDP\D{0,30}?(\d{2,4})\s*W', text, re.IGNORECASE)

                if length_match and width_match and height_match:
                    result["dimensions"] = {
                        "length_mm": float(length_match.group(1).replace(',', '.')),
                        "width_mm": float(width_match.group(1).replace(',', '.')),
                        "height_mm": float(height_match.group(1).replace(',', '.'))
                    }
                    result["status"] = "Sikeres"

                if tdp_match:
                    result["tdp"] = {"value": int(tdp_match.group(1)), "unit": "W"}
                    if result["status"] == "Folyamatban":
                        result["status"] = "Sikeres (Csak TDP)"

            # Kilépés, ha sikeres
            if result["status"] != "Folyamatban":
                return result

        except Exception as e:
            result["status"] = f"Scraper hiba: {str(e)}"

        # Várakozás
        if attempt < max_retries - 1:
            time.sleep(2)

    result["status"] = f"Hiba: {max_retries} próbálkozás után sem sikerült kinyerni az adatokat (Utolsó hiba: {result['status']})."
    return result

 ## CPU scraper

In [ ]:
def get_CPU_specs(component_name: str, max_retries: int = 3) -> dict:
    """Kikeresi az asztali processzor (CPU) adatait a TechPowerUp adatbázisából, beépített újrapróbálkozással."""

    for attempt in range(max_retries):
        result = {"component_name": component_name, "url": None, "tdp": None, "socket": None, "status": "Folyamatban"}

        try:
            # 1. Keresési kifejezés
            search_query = f"{component_name} site:techpowerup.com/cpu-specs -mobile -laptop"
            web_search = TavilySearch(max_results=5)
            search_results = web_search.invoke(search_query)

            # 2. URL szűrő
            def is_valid_desktop_url(url):
                url_lower = url.lower()
                if "techpowerup.com/cpu-specs/" not in url_lower:
                    return False
                if any(keyword in url_lower for keyword in ["-mobile", "-laptop"]):
                    return False
                return True

            target_url = next((res["url"] for res in search_results.get("results", []) if is_valid_desktop_url(res["url"])), None)

            if not target_url:
                result["status"] = "Hiba: Nem találtam megfelelő asztali processzor adatlapot."
                return result

            result["url"] = target_url

            # 3. ScraperAPI hívás
            payload = {'api_key': SCRAPER_API_KEY, 'url': target_url, 'render': 'false', 'premium': 'true'}
            response = requests.get('https://api.scraperapi.com/', params=payload, timeout=60)

            if response.status_code != 200:
                result["status"] = f"HTTP Hiba: {response.status_code}"
            else:
                soup = BeautifulSoup(response.text, 'html.parser')
                for s_tag in soup.find_all(['s', 'del']): s_tag.decompose()
                for element in soup(["script", "style", "nav", "footer", "header"]): element.decompose()
                text = soup.get_text(separator=' ', strip=True)

                tdp_match = re.search(r'TDP\D{0,30}?(\d{2,4})\s*W', text, re.IGNORECASE)
                socket_match = re.search(r'Socket\W{0,30}?([A-Za-z0-9\-]+)', text, re.IGNORECASE)

                if tdp_match:
                    result["tdp"] = int(tdp_match.group(1))
                    result["status"] = "Sikeres"
                if socket_match:
                    result["socket"] = socket_match.group(1).strip()
                    if result["status"] == "Folyamatban":
                        result["status"] = "Sikeres (Csak Socket)"

            if result["status"] != "Folyamatban":
                return result

        except Exception as e:
            result["status"] = f"Scraper hiba: {str(e)}"

        if attempt < max_retries - 1:
            time.sleep(2)

    result["status"] = f"Hiba: {max_retries} próbálkozás után sem sikerült kinyerni az adatokat (Utolsó hiba: {result['status']})."
    return result

#Segéd toolok az adatároláshoz


In [ ]:
'''
#ELAVULT
def add_component_to_build(type: str, name: str, tdp: int = None, length: float = None, width: float = None, height: float = None) -> str:
    """
    Hozzáad egy alkatrészt az aktuális PC építéshez.
    Használd ezt a toolt, miután megszerezted az alkatrész adatait (név, TDP, méretek).
    """
    dims = None
    if length and width and height:
        dims = {"length_mm": length, "width_mm": width, "height_mm": height}

    component = PCComponent(type=type, name=name, tdp=tdp, dimensions=dims)
    return update_build_database(component)
'''
#ÚJ
def add_component_to_build(
    name: str,
    type: str,
    tdp: Optional[int] = None,
    length_mm: Optional[float] = None,
    width_mm: Optional[float] = None,
    height_mm: Optional[float] = None,
    socket: Optional[str] = None
) -> str:
    """
    Hozzáad egy új alkatrészt a jelenlegi PC buildhez, vagy felülírja a meglévőt (ha azonos a típusa).

    KÖTELEZŐ paraméterek: name, type ('GPU', 'CPU', 'Motherboard', stb.)
    OPCIONÁLIS paraméterek: tdp, length_mm, width_mm, height_mm, socket. Csak a releváns adatokat add át!
    """

    # Például, ha mindent nagybetűsítünk a biztonság kedvéért (GPU, CPU)
    comp_type = type.strip().upper()

    # Létrehozzuk az új komponenst az opcionális paraméterekkel
    new_component = PCComponent(
        name=name,
        type=comp_type,
        tdp=tdp,
        length_mm=length_mm,
        width_mm=width_mm,
        height_mm=height_mm,
        socket=socket
    )

    # Betesszük a globális szótárba (adatbázisba)
    current_pc_build[comp_type] = new_component

    # Visszajelzés az LLM-nek, hogy tudja, mi történt
    added_info = f"Sikeresen hozzáadva: {comp_type} -> {name}."
    if tdp:
        added_info += f" (TDP: {tdp}W)"
    if socket:
        added_info += f" (Socket: {socket})"

    return added_info

In [ ]:
'''
#ELAVULT
def update_build_database(component: PCComponent) -> str:
    """Hozzáad vagy frissít egy alkatrészt az adatbázisban."""
    # Mivel a kulcs a típus (pl. "GPU"), a régi kártya automatikusan felülíródik
    current_pc_build[component.type] = component
    return f"Sikeresen frissítve a gépben: {component.type} -> {component.name}"
'''

'\n#ELAVULT\ndef update_build_database(component: PCComponent) -> str:\n    """Hozzáad vagy frissít egy alkatrészt az adatbázisban."""\n    # Mivel a kulcs a típus (pl. "GPU"), a régi kártya automatikusan felülíródik\n    current_pc_build[component.type] = component\n    return f"Sikeresen frissítve a gépben: {component.type} -> {component.name}"\n'

In [ ]:
def clear_build_database() -> str:
    """
    Törli az összes alkatrészt az aktuális gépből.
    Használd ezt, ha a felhasználó új gépet akar kezdeni, vagy mindent törölni akar.
    """
    current_pc_build.clear()
    return "A gép sikeresen kiürítve. Minden alkatrész törölve lett az adatbázisból."

In [ ]:
def remove_component_from_build(component_type: str) -> str:
    """
    Eltávolít egy specifikus komponenst (pl. 'GPU', 'CPU', 'Motherboard') a gépből.
    """
    # A biztonság kedvéért a típust standardizáljuk
    comp_type = component_type.strip().upper()

    # Keresés a szótárban (kulcsok lehetnek eltérő case-ek, pl "GPU" vs "Gpu")
    keys_to_remove = [k for k in current_pc_build.keys() if k.upper() == comp_type]

    if keys_to_remove:
        for k in keys_to_remove:
            del current_pc_build[k]
        return f"Sikeresen eltávolítva a gépből a következő típus: {comp_type}"
    else:
        return f"Hiba: Nem találtam '{comp_type}' típusú alkatrészt a jelenlegi buildben."

#Valuta váltó
ExchangeRate-API: Valós idejű árfolyamok lekérése kulcs nélkül.  

Multi-currency: Automatikus konverzió USD, EUR, HUF és RSD valutákba.

In [ ]:
def calculate_prices(amount: float, base_currency: str = "USD") -> dict:
    """
    Kiszámítja egy termék árát több különböző valutában (EUR, HUF, RSD, USD)
    az aktuális, valós idejű árfolyamok alapján.
    Használd ezt az eszközt, ha árakat kell átszámítani a komponensekhez.
    """
    result = {
        "base_amount": amount,
        "base_currency": base_currency.upper(),
        "converted_prices": {},
        "status": "Folyamatban"
    }

    # Ezeket a valutákat kérjük le mindenképpen
    target_currencies = ["EUR", "HUF", "RSD", "USD"]

    # Ha a bázis valuta is benne van a célok között, az maga az eredeti összeg
    result["converted_prices"][base_currency.upper()] = amount

    try:
        # Az ExchangeRate-API nyílt, kulcs nélküli végpontja
        url = f"https://open.er-api.com/v6/latest/{base_currency.upper()}"
        response = requests.get(url, timeout=10)

        if response.status_code != 200:
            result["status"] = f"Hálózati hiba. HTTP Státusz: {response.status_code}"
            return result

        data = response.json()

        if "rates" not in data:
            result["status"] = "Hiba: Az API válasza nem tartalmaz árfolyamadatokat."
            return result

        rates = data["rates"]

        # Végigmegyünk a kért valutákon, és kiszámoljuk az értéket
        for currency in target_currencies:
            if currency != base_currency.upper():
                if currency in rates:
                    # Kerekítés: HUF és RSD egész szám, EUR és USD 2 tizedes
                    calculated_value = amount * rates[currency]
                    if currency in ["HUF", "RSD"]:
                        result["converted_prices"][currency] = int(round(calculated_value, 0))
                    else:
                        result["converted_prices"][currency] = round(calculated_value, 2)
                else:
                    result["converted_prices"][currency] = "N/A"

        result["status"] = "Sikeres átváltás"

    except requests.exceptions.Timeout:
        result["status"] = "Hiba: Időtúllépés az árfolyam API hívásakor."
    except Exception as e:
        result["status"] = f"API feldolgozási hiba: {str(e)}"

    return result

#TDP kalkulátor
Mérnöki számítás: Összegzi a build TDP-jét, hozzáad 50W alapfogyasztást és 20% biztonsági tartalékot.  

Hibaellenőrzés: Jelez az ágensnek, ha valamelyik alkatrész fogyasztása hiányzik a pontos számításhoz.

In [ ]:
def calculate_system_psu() -> str:
    """
    Kiszámítja az aktuális gépházban (current_pc_build) lévő alkatrészek alapján
    az ajánlott tápegység (PSU) méretét.
    Ezt a toolt akkor használd, ha a felhasználó megkérdezi, mekkora táp kell a gépbe.
    """
    if not current_pc_build:
        return "A gép jelenleg üres, nem tudok tápegységet számolni."

    total_tdp = 0
    missing_tdp_components = []

    # 1. Összeadjuk a megismert TDP-ket
    for comp_type, comp in current_pc_build.items():
        if comp.tdp is not None:
            total_tdp += comp.tdp
        else:
            missing_tdp_components.append(comp.name)

    # 2. Alaplap, RAM, SSD és ventilátorok alapfogyasztásának hozzáadása (kb. 50W)
    base_system_draw = 50
    total_draw = total_tdp + base_system_draw

    # 3. Biztonsági tartalék (20%)
    recommended_psu = total_draw * 1.2

    # --- Válasz összeállítása az ágensnek ---
    response = (
        f"A rendszer becsült csúcsfogyasztása: {total_draw}W "
        f"(Alkatrészek: {total_tdp}W + Alaprendszer: {base_system_draw}W).\n"
        f"A 20% biztonsági tartalékot rászámolva az AJÁNLOTT TÁPEGYSÉG: minimum {round(recommended_psu)}W."
    )

    # Ha volt olyan alkatrész, aminek nem tudjuk a fogyasztását, figyelmeztetjük az ágenst
    if missing_tdp_components:
        response += f"\n\nFIGYELEM: A következő alkatrészek TDP értéke hiányzik az adatbázisból, így a számítás pontatlan lehet: {', '.join(missing_tdp_components)}."

    return response

# Fórum Sentiment Elemzés (Hugging Face)
A közösségi vélemények (Reddit) elemzéséhez a hivatalos `huggingface_hub` könyvtár `InferenceClient`-jét használjuk explicit token átadással (`HF_TOKEN`). Ez kiküszöböli a manuális REST API hívásoknál (requests.post) fellépő 404-es routing hibákat és a jogosultsági problémákat. A választott modell (`lxyuan/distilbert-...-student`) kifejezetten az ingyenes Serverless Inference API-ra van optimalizálva, garantálva a magas rendelkezésre állást.

In [ ]:
def get_forum_sentiment(component_name: str) -> str:
    """
    Reddit vélemények alapján hangulatelemzést végez a Hugging Face Inference API segítségével.
    """
    try:
        # 1. Reddit vélemények lekérése a Tavily-vel
        web_search = TavilySearch(max_results=3)
        search_results = web_search.invoke(f"site:reddit.com {component_name} review opinion")
        snippets = [res["content"] for res in search_results.get("results", [])] if search_results else []

        if not snippets:
            return "Nem találtam elég Reddit véleményt az elemzéshez."

        combined_text = " ".join(snippets)[:1000]

        # 2. Kapcsolódás az AI modellhez a HF_TOKEN használatával
        client = InferenceClient(token=os.environ.get("HF_TOKEN"))

        # Ez a modell stabil és ingyenesen elérhető az API-n keresztül
        model_id = "lxyuan/distilbert-base-multilingual-cased-sentiments-student"

        response = client.text_classification(combined_text, model=model_id)

        # 3. Eredmény formázása
        best_result = response[0]
        label = best_result['label'].upper()
        score = best_result['score']

        if "POSITIVE" in label:
            magyar_label = "POZITÍV 🟢"
        elif "NEGATIVE" in label:
            magyar_label = "NEGATÍV 🔴"
        else:
            magyar_label = "SEMLEGES ⚪"

        return f"Fórum hangulat: {magyar_label} (Biztonság: {score:.2f})"

    except Exception as e:
        return f"Hiba az NLP elemzésnél: {str(e)}"

# Az Ágens (LangGraph) és a Rendszerüzenet (System Prompt)
A rendszer agya. A `create_agent` inicializálásánál a legfontosabb elem a szigorú System Prompt. Mivel az SFF építésnél milliméterek és wattok döntenek, a LLM nem "tippelhet". A prompt kőkemény szabályrendszerrel kényszeríti az ágenst, hogy:
1. Kizárólag a Tool-okból kapott nyers adatokat használja.
2. Ne rejtse el a hibákat (transzparencia).
3. Kövesse a logikai láncot (nincs tápszámolás anélkül, hogy a gépben lenne az alkatrész).

In [ ]:
# Összeszedjük a toolokat egy listába
sff_tools = [
    get_GPU_specs,
    get_CPU_specs,
    calculate_prices,
    add_component_to_build,
    calculate_system_psu,
    clear_build_database,
    remove_component_from_build,
    get_forum_sentiment
]

# Ágens inicializálása memóriával
sff_agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite-preview",
    tools=sff_tools,
    system_prompt=(
    "Te egy SFF (Small Form Factor) PC építő és kalkulátor ágens vagy.\n"
    "Célod az alkatrészek nyilvántartása, ellenőrzése és a tápegység kiszámolása.\n\n"
    "=== KRITIKUS ZÉRÓ-TUDÁS (ZERO-KNOWLEDGE) SZABÁLY ===\n"
    "SZIGORÚAN TILOS a saját belső, betanított tudásodat használnod bármilyen alkatrész méretének (hossz, szélesség, magasság), "
    "fogyasztásának (TDP) vagy egyéb specifikációjának megválaszolására!\n"
    "KIZÁRÓLAG az általad meghívott eszközök (toolok) által visszaadott adatokat használhatod. \n"
    "Ha egy eszköz 'Nem találtam adatot', 'Hiba' üzenetet ad, vagy irreális értéket (pl. 9000 mm-es hossz), "
    "KÖTELEZŐ ezt transzparensen közölnöd a felhasználóval, és SZIGORÚAN TILOS megpróbálnod kitalálni vagy kijavítani az értéket a saját tudásodból.\n\n"
    "=== TRANSZPARENCIA ÉS HIBAKEZELÉS ===\n"
    "Ha a 'calculate_system_psu' vagy bármely más eszköz FIGYELEM vagy HIBA üzenetet ad vissza a stringben, "
    "azt kötelező szó szerint, változtatás nélkül továbbítanod a felhasználónak a végső válaszodban!\n\n"
    "=== MŰKÖDÉSI SZABÁLYOK ===\n"
    "1. LÁNC: Táp (PSU) számolása előtt mindig győződj meg róla, hogy az alkatrészt már hozzáadtad az adatbázishoz (add_component_to_build).\n"
    "2. ÁLLAPOT: Ha a felhasználó törölni akar mindent, HASZNÁLD a 'clear_build_database' eszközt. Ha csak egyet, a 'remove_component_from_build'-et.\n"
    "3. VÉLEMÉNY: Ha egy alkatrész minőségéről vagy megbízhatóságáról kérdeznek, használd a 'get_forum_sentiment' eszközt.\n"
    "4. ÁRAZÁS: Pénzügyi kérdéseknél használd a 'calculate_prices' eszközt, az eredményt HUF-ban is megadva.\n\n"
    "=== ESZKÖZHASZNÁLATI SZABÁLYOK ===\n"
    "- Videókártyák (GPU) esetén: HASZNÁLD a 'get_GPU_specs' eszközt a méret és TDP kinyerésére.\n"
    "- Processzorok (CPU) esetén: KIZÁRÓLAG a 'get_CPU_specs' eszközt használd a TDP és a foglalat (Socket) lekérdezéséhez.\n"
    ),
    name="sff_builder_agent",
    checkpointer=InMemorySaver()
)

In [ ]:
def pretty_print_message(message, indent=False):
    """Üzenetek formázott kiírása, a Gemini listás formátumának okos kezelésével."""
    # A szerepkör (Role) formázása fejlécként
    role = message.__class__.__name__.replace("Message", "")
    header = f"========== {role} =========="

    # 1. Szöveges tartalom kinyerése és tisztítása
    content = message.content
    display_text = ""

    if isinstance(content, list):
        # Ha a Gemini egy listányi dictionary-t ad vissza
        text_parts = [item['text'] for item in content if isinstance(item, dict) and 'text' in item]
        display_text = "\n".join(text_parts)
    else:
        # Ha normál string
        display_text = str(content)

    # 2. Eszközhasználat (Tool Calls) szép kiírása
    if hasattr(message, 'tool_calls') and message.tool_calls:
        if display_text.strip():
            display_text += "\n\n"
        display_text += "[⚙️ ESZKÖZHASZNÁLAT]:\n"
        for tc in message.tool_calls:
            display_text += f"  🔧 {tc.get('name')} | Args: {tc.get('args')}\n"

    # Összefűzés
    pretty_message = f"\n{header}\n{display_text.strip()}\n"

    # Behúzás (indent) kezelése subgraph-oknál
    if not indent:
        print(pretty_message)
        return
    indented = "\n".join("\t" + c for c in pretty_message.split("\n"))
    print(indented)


def pretty_print_messages(update, last_message=False):
    """Ágens frissítések és eszközhívások vizualizálása a futás során."""
    is_subgraph = False
    if isinstance(update, tuple):
        ns, update = update
        if len(ns) == 0:
            return
        graph_id = ns[-1].split(":")[0]
        print(f"\nUpdate from subgraph {graph_id}:")
        is_subgraph = True

    for node_name, node_update in update.items():
        update_label = f"\n[ Csomópont: {node_name.upper()} ]"
        if is_subgraph:
            update_label = "\t" + update_label
        print(update_label)

        messages = convert_to_messages(node_update["messages"])
        if last_message:
            messages = messages[-1:]

        for m in messages:
            pretty_print_message(m, indent=is_subgraph)

In [ ]:
# Konfiguráció a memóriához (thread_id azonosítja a beszélgetést)
config = {
    "configurable": {
        "thread_id": "sff_build_session_1"
    }
}

# 1. Parancs: Adatgyűjtés és rögzítés
prompt_1 = "Szeretnék venni egy Asus RTX 4070-et. Nézd meg a méreteit és a TDP-jét, majd add hozzá a gépemhez!"
print("--- 1. PROMPT KÜLDÉSE ---")
for chunk in sff_agent.stream({"messages": [{"role": "user", "content": prompt_1}]}, config=config):
    pretty_print_messages(chunk)

# 2. Parancs: Matek és Kalkuláció a memória alapján
prompt_2 = "Adj a géphez egy Ryzen 7 9800X3D processzort (120W TDP). Ezek alapján mekkora tápegységet vegyek?"
print("\n--- 2. PROMPT KÜLDÉSE ---")
for chunk in sff_agent.stream({"messages": [{"role": "user", "content": prompt_2}]}, config=config):
    pretty_print_messages(chunk)

--- 1. PROMPT KÜLDÉSE ---

[ Csomópont: MODEL ]

========== AI ==========
[⚙️ ESZKÖZHASZNÁLAT]:
  🔧 get_GPU_specs | Args: {'component_name': 'Asus RTX 4070'}


[ Csomópont: TOOLS ]

========== Tool ==========
{"component_name": "Asus RTX 4070", "url": "https://www.techpowerup.com/gpu-specs/geforce-rtx-4070.c3924?ref=GBXBlP&e=&rurl=https://www.techpowerup.com/gpu-specs/geforce-rtx-4070.c3924", "dimensions": {"length_mm": 240.0, "width_mm": 40.0, "height_mm": 110.0}, "tdp": {"value": 200, "unit": "W"}, "status": "Sikeres"}


[ Csomópont: MODEL ]

========== AI ==========
[⚙️ ESZKÖZHASZNÁLAT]:
  🔧 add_component_to_build | Args: {'name': 'Asus RTX 4070', 'type': 'GPU', 'length_mm': 240, 'height_mm': 110, 'tdp': 200, 'width_mm': 40}


[ Csomópont: TOOLS ]

========== Tool ==========
Sikeresen hozzáadva: GPU -> Asus RTX 4070. (TDP: 200W)


[ Csomópont: MODEL ]

========== AI ==========
Az Asus RTX 4070 sikeresen hozzáadásra került a gépedhez. Íme az adatai:

*   **Méretek:** 240 mm (hossz) x

#Unit tesztek

In [ ]:
import time

print("\n" + "="*70)
print("🚀 SFF BUILDER RENDSZER - TELJESKÖRŰ UNIT TESZTEK INDÍTÁSA")
print("="*70)

# ---------------------------------------------------------
# 1. TESZT: GPU Scraper (TechPowerUp)
# ---------------------------------------------------------
test_gpu = "Asus Dual EVO OC GeForce RTX 4070"
print(f"\n[1/5] 🎮 GPU Adatgyűjtés tesztelése: {test_gpu}")
start_time = time.time()
gpu_result = get_GPU_specs(test_gpu) # Használd a pontos függvénynevedet!
exec_time = time.time() - start_time

print(f"  ⏱️ Futási idő: {exec_time:.2f} s")
print(f"  🔗 URL: {gpu_result.get('url')}")
print(f"  📊 Státusz: {gpu_result.get('status')}")
print(f"  📏 Dimenziók: {gpu_result.get('dimensions')}")
print(f"  ⚡ TDP: {gpu_result.get('tdp')}")

# ---------------------------------------------------------
# 2. TESZT: CPU Scraper (TechPowerUp)
# ---------------------------------------------------------
test_cpu = "AMD Ryzen 7 9800X3D"
print(f"\n[2/5] 🧠 CPU Adatgyűjtés tesztelése: {test_cpu}")
start_time = time.time()
cpu_result = get_CPU_specs(test_cpu) # Használd a pontos függvénynevedet!
exec_time = time.time() - start_time

print(f"  ⏱️ Futási idő: {exec_time:.2f} s")
print(f"  🔗 URL: {cpu_result.get('url')}")
print(f"  📊 Státusz: {cpu_result.get('status')}")
print(f"  ⚡ TDP: {cpu_result.get('tdp')}")
print(f"  🔌 Foglalat (Socket): {cpu_result.get('socket')}")

# ---------------------------------------------------------
# 3. TESZT: Forum Sentiment (Reddit + Hugging Face)
# ---------------------------------------------------------
test_part = "ASRock B650I Lightning WiFi motherboard"
print(f"\n[3/5] 💬 Fórum Sentiment tesztelése: {test_part}")
start_time = time.time()
sentiment_result = get_forum_sentiment(test_part)
exec_time = time.time() - start_time

print(f"  ⏱️ Futási idő: {exec_time:.2f} s")
print(f"  🤖 Eredmény:\n  {sentiment_result}")

# ---------------------------------------------------------
# 4. TESZT: Valutaváltó (ExchangeRate-API)
# ---------------------------------------------------------
test_price = 599.99
base_curr = "USD"
print(f"\n[4/5] 💱 Valutaváltó tesztelése: {test_price} {base_curr}")
start_time = time.time()
price_result = calculate_prices(test_price, base_curr)
exec_time = time.time() - start_time

print(f"  ⏱️ Futási idő: {exec_time:.2f} s")
print(f"  📊 Státusz: {price_result.get('status')}")
if "converted_prices" in price_result:
    for curr, val in price_result["converted_prices"].items():
        print(f"    - {curr}: {val}")

# ---------------------------------------------------------
# 5. TESZT: Állapotkezelés (State) és PSU Kalkulátor Logic
# ---------------------------------------------------------
print("\n[5/5] 🛠️ Állapotkezelés és Logika tesztelése...")

# A) Tisztítjuk a gépet (hogy tiszta lappal induljunk)
print("  🧹 1. Lépés:", clear_build_database())

# B) Hozzáadunk egy CPU-t
print("  ➕ 2. Lépés:", add_component_to_build(
    name="AMD Ryzen 7 9800X3D",
    type="CPU",
    tdp=120,
    socket="AM5"
))

# C) Hozzáadunk egy GPU-t méretekkel
print("  ➕ 3. Lépés:", add_component_to_build(
    name="Asus RTX 4070",
    type="GPU",
    tdp=220,
    length_mm=269.0,
    width_mm=120.0,
    height_mm=50.0
))

# D) Számolunk egy tápot az aktuális gépre
print("\n  🧮 4. Lépés: PSU Kalkuláció (CPU + GPU)")
print("  " + calculate_system_psu().replace("\n", "\n  "))

# E) Eltávolítjuk a GPU-t
print("\n  ➖ 5. Lépés:", remove_component_from_build("GPU"))

# F) Újra számolunk tápot (most már csak a CPU van benne)
print("\n  🧮 6. Lépés: PSU Kalkuláció (Csak CPU)")
print("  " + calculate_system_psu().replace("\n", "\n  "))

# G) Végső tisztítás
print("\n  🧹 7. Lépés (Befejezés):", clear_build_database())

print("\n" + "="*70)
print("✅ UNIT TESZTEK SIKERESEN BEFEJEZŐDTEK!")
print("="*70)


🚀 SFF BUILDER RENDSZER - TELJESKÖRŰ UNIT TESZTEK INDÍTÁSA

[1/5] 🎮 GPU Adatgyűjtés tesztelése: Asus Dual EVO OC GeForce RTX 4070
  ⏱️ Futási idő: 12.10 s
  🔗 URL: https://www.techpowerup.com/gpu-specs/asus-dual-rtx-4070-evo-gddr6.b11901
  📊 Státusz: Sikeres
  📏 Dimenziók: {'length_mm': 227.0, 'width_mm': 50.0, 'height_mm': 123.0}
  ⚡ TDP: {'value': 200, 'unit': 'W'}

[2/5] 🧠 CPU Adatgyűjtés tesztelése: AMD Ryzen 7 9800X3D
  ⏱️ Futási idő: 4.86 s
  🔗 URL: https://www.techpowerup.com/cpu-specs/ryzen-7-9800x3d.c3891
  📊 Státusz: Sikeres
  ⚡ TDP: 120
  🔌 Foglalat (Socket): AM5

[3/5] 💬 Fórum Sentiment tesztelése: ASRock B650I Lightning WiFi motherboard


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


  ⏱️ Futási idő: 6.77 s
  🤖 Eredmény:
  Fórum hangulat: POZITÍV 🟢 (Biztonság: 0.58)

[4/5] 💱 Valutaváltó tesztelése: 599.99 USD
  ⏱️ Futási idő: 0.11 s
  📊 Státusz: Sikeres átváltás
    - USD: 599.99
    - EUR: 511.67
    - HUF: 186517
    - RSD: 60100

[5/5] 🛠️ Állapotkezelés és Logika tesztelése...
  🧹 1. Lépés: A gép sikeresen kiürítve. Minden alkatrész törölve lett az adatbázisból.
  ➕ 2. Lépés: Sikeresen hozzáadva: CPU -> AMD Ryzen 7 9800X3D. (TDP: 120W) (Socket: AM5)
  ➕ 3. Lépés: Sikeresen hozzáadva: GPU -> Asus RTX 4070. (TDP: 220W)

  🧮 4. Lépés: PSU Kalkuláció (CPU + GPU)
  A rendszer becsült csúcsfogyasztása: 390W (Alkatrészek: 340W + Alaprendszer: 50W).
  A 20% biztonsági tartalékot rászámolva az AJÁNLOTT TÁPEGYSÉG: minimum 468W.

  ➖ 5. Lépés: Sikeresen eltávolítva a gépből a következő típus: GPU

  🧮 6. Lépés: PSU Kalkuláció (Csak CPU)
  A rendszer becsült csúcsfogyasztása: 170W (Alkatrészek: 120W + Alaprendszer: 50W).
  A 20% biztonsági tartalékot rászámolva az AJÁNLOTT 

# Rendszer Értékelése és Hibaellenőrzés


In [ ]:
test_bank = [
    # Egyszerű Adatgyűjtés
    ("Single Tool", "1. Teszt", "Mekkorák az Asus RTX 4070 méretei?"),
    ("Single Tool", "2. Teszt", "Mennyi a Ryzen 7 9800X3D fogyasztása?"),
    ("Single Tool", "3. Teszt", "Válts át 650 dollárt forintra!"),

    # Kombinált Adatgyűjtés
    ("Multi-Tool", "4. Teszt", "Keresd meg egy MSI RTX 4090 méreteit és a TDP-jét is!"),
    ("Multi-Tool", "5. Teszt", "Nézd meg a Sapphire Pulse RX 7800 XT adatait, és számold ki egy 550 eurós árból a forintot!"),

    # Memória és Állapotkezelés (Innentől fontos a state!)
    ("State Management", "6. Teszt", "Add hozzá a gépemhez a Ryzen 7 9800X3D-t!"),
    ("State Management", "7. Teszt", "Add hozzá a korábban keresett MSI RTX 4090-et is!"),
    ("State Management", "8. Teszt", "Cseréld le a videókártyát egy Asus RTX 4070-re!"),
    ("State Management", "9. Teszt", "Mik vannak most a gépemben?"),

    # Logika és Kalkulációk
    ("Logic", "10. Teszt", "Számold ki, mekkora tápegység kell a jelenlegi gépemhez!"),
    ("Logic", "11. Teszt", "Adj hozzá egy ASRock B650I alaplapot, majd számolj újra tápot!"),
    ("Logic", "12. Teszt", "Mennyibe kerülne ez a gép, ha a CPU 400 USD, a GPU pedig 600 EUR?"),

    # Komplex, Implicit Kérések (Edge Cases)
    ("Edge Case", "13. Teszt", "Ez a kártya befér egy 300mm-es helyre: Gigabyte RX 7600?"),
    ("Edge Case", "14. Teszt", "Szeretnék egy Gigabyte RTX 4080-at a gépbe. Mekkora tápot vegyek?"),
    ("Edge Case", "15. Teszt", "Elég lesz egy 500W-os táp a gépemhez?"),
    ("Edge Case", "16. Teszt", "Törölj mindent a gépből!"),

    # Hibageneráló / Adversarial Promptek
    ("Adversarial", "17. Teszt", "Mekkorák egy Nvidia RTX 9090 Ti méretei?"),
    ("Adversarial", "18. Teszt", "Nézd meg a PistaBá Custom GPU adatait!"),
    ("Adversarial", "19. Teszt", "Adj hozzá egy kártyát 5000W TDP-vel!")
]
new_hardcore_tests = [
    # Halálos "Silent Failure" teszt (Kikényszeríti az irreális adat felismerését)
    ("Strict Enforcement", "20. Teszt", "Keresd meg a Radeon RX 7600 adatait! Ha irreális méretet kapsz, semmiképp ne próbáld meg kitalálni a valósat, csak mondd meg, mit adott vissza a gép!"),

    # Komplex fizikai és áramellátási logika
    ("Physical Constraints", "21. Teszt", "Egy nagyon szűk, 9 literes, kétkamrás házam van, ahol a két oldal külön rekeszben van, és közvetlenül az oldalsó rácsokon keresztül szívják be a levegőt. Egy Ryzen 7 9800X3D-t tervezek bele egy alacsony profilú hűtővel. Add hozzá a CPU-t a géphez, és mondd meg, milyen hűtési buktatókra számíthatok egy ilyen közvetlen légáramlásos, rekeszes házban!"),

    # Állapotkezelés megzavarása (Adversarial State Injection)
    ("State Manipulation", "22. Teszt", "Ürítsd ki a gépet, majd add hozzá a 120W-os Ryzen 7 9800X3D-t! Ezután NE számolj tápot, csak adj egy Fórum sentiment elemzést az 'ASRock B650I Lightning' alaplapról!"),

    # Zéró-Tudás megsértésének kísérlete
    ("Zero-Knowledge Trap", "23. Teszt", "Mekkora a hossza az Asus Dual EVO OC GeForce RTX 4070-nek? A tool használata nélkül mondd meg, ha tudod!"),
]


def run_test_suite(agent, tests, thread_id="comprehensive_test_session_01"):
    """
    Végigfuttatja a teljes tesztbankot az ágensen, megőrizve a memóriát.
    """
    current_pc_build.clear()

    config = {"configurable": {"thread_id": thread_id}}
    for category, test_name, prompt in tests:
        print(f"\n\n{'='*70}")
        print(f"[ {category} ] - {test_name}")
        print(f"USER PROMPT: {prompt}")
        print(f"{'='*70}")

        try:
            for chunk in agent.stream({"messages": [{"role": "user", "content": prompt}]}, config=config):
                pretty_print_messages(chunk)

        except Exception as e:
            print(f"\n❌ KRITIKUS HIBA A FUTTÁS SORÁN: {str(e)}")

        time.sleep(5)

run_test_suite(sff_agent, test_bank)
run_test_suite(sff_agent, new_hardcore_tests)



[ Single Tool ] - 1. Teszt
USER PROMPT: Mekkorák az Asus RTX 4070 méretei?

[ Csomópont: MODEL ]

========== AI ==========
[⚙️ ESZKÖZHASZNÁLAT]:
  🔧 get_GPU_specs | Args: {'component_name': 'Asus RTX 4070'}


[ Csomópont: TOOLS ]

========== Tool ==========
{"component_name": "Asus RTX 4070", "url": "https://www.techpowerup.com/gpu-specs/geforce-rtx-4070.c3924?ref=GBXBlP&e=&rurl=https://www.techpowerup.com/gpu-specs/geforce-rtx-4070.c3924", "dimensions": {"length_mm": 240.0, "width_mm": 40.0, "height_mm": 110.0}, "tdp": {"value": 200, "unit": "W"}, "status": "Sikeres"}


[ Csomópont: MODEL ]

========== AI ==========
Az Asus RTX 4070 méretei az adatbázis szerint a következők:
*   Hosszúság: 240 mm
*   Szélesség: 40 mm
*   Magasság: 110 mm



[ Single Tool ] - 2. Teszt
USER PROMPT: Mennyi a Ryzen 7 9800X3D fogyasztása?

[ Csomópont: MODEL ]

========== AI ==========
[⚙️ ESZKÖZHASZNÁLAT]:
  🔧 get_CPU_specs | Args: {'component_name': 'Ryzen 7 9800X3D'}


[ Csomópont: TOOLS ]

==========



---



A feladat során összesen 23 kérdésből álló, komplex tesztbankkal vizsgáltam meg a LangGraph alapú ágens működését. A tesztek lefedték a szimpla adatgyűjtést, a több eszközt igénylő láncolatokat, a memória-állapotkezelést (State Management), az extrém eseteket (Edge Cases) és a szándékosan félrevezető (Adversarial) parancsokat is.

A fejlesztés iteratív volt: a kezdeti tesztek rávilágítottak a rendszer gyenge pontjaira, amelyeket folyamatos prompt- és architekturális finomhangolással javítottam.

## 1. Milyen esetekben rontott az ágens (Kezdeti hibák és a megoldásuk)?

Az első tesztkörök során a nagy nyelvi modellek (LLM-ek) tipikus hibáival szembesültem, amelyeket a végső verzióban sikeresen kiküszöböltem:

*   **Eszközök önkényes mellőzése (Zéró-tudás megsértése):**
    *   *A probléma:* Abszurd vagy nem létező hardvereknél ("Nvidia RTX 9090 Ti") az ágens nem használta a keresőt, hanem saját lexikális tudásából azonnal elutasította a kérést. Ez egy adatvezérelt ágensnél szabályszegés.
    *   *A megoldás:* Bevezettem egy szigorú, "IF-THEN" alapú *System Promptot*, ami kényszeríti az ágenst, hogy a válaszadás előtt kötelezően hívja meg a toolt. A tesztek alapján az ágens immár vakon le is futtatja a keresést a lehetetlen kérésekre is.
*   **"Silent Failure" (Csendes Hiba) és Hallucináció:**
    *   *A probléma:* Ha a scraper irreális adatokat hozott (pl. a Regex egy 7600-as modellszámot nézett hossznak), az ágens észlelte a hibát, eldobta az adatot, és a saját súlyaiból (weights) behallucinálta a valós méreteket.
    *   *A megoldás:* Szigorítottam a promptot a transzparenciára vonatkozóan, és bevezettem a V2-es Scrapert, ami kizárja a mobil chipeket a keresésből (`-mobile`, `-laptop`), így letisztítva az LLM elé kerülő adatokat.
*   **Elrejtett figyelmeztetések:**
    *   *A probléma:* A PSU kalkulátor figyelmeztetését a hiányzó TDP adatokról az ágens eltitkolta a felhasználó elől.
    *   *A megoldás:* Kényszerített transzparencia a promptban. A 11. és 14. tesztek bizonyítják, hogy most már vastag betűvel, szó szerint továbbítja a `FIGYELEM` üzeneteket.
*   **Állapotkezelési (State) anomáliák:**
    *   *A probléma:* A korábbi tesztek adatai "átszivárogtak" az újakba (State Bleed), illetve a törlés parancsra az ágens csak egy fiktív "Empty build" komponenst adott a listához.
    *   *A megoldás:* Létrehoztam egy natív `clear_build_database` toolt, és a tesztkörnyezetet dedikált `thread_id`-kkel szigeteltem el.

## 2. Mennyire volt stabil az API-k válasza?

A külső rendszerek megbízhatósága vegyes képet mutatott, de a robusztus kódolással stabilizálható volt:

*   **Valutaváltó (ExchangeRate-API):** Kiemelkedően stabil és gyors. Minden teszt (3., 5., 12.) hiba nélkül lefutott.
*   **Fórum Sentiment (Hugging Face Inference API):** Kezdetben instabil volt HTTP 404 és jogosultsági hibák miatt. A hivatalos `huggingface_hub` klienskönyvtár és a `HF_TOKEN` explicit használata után 100%-os stabilitást ért el.
*   **Hardver Scraper (Tavily + ScraperAPI + TechPowerUp):** Ez volt a leggyengébb láncszem a weboldalak bot-védelme (Cloudflare) miatt. A megoldást az hozta el, hogy a ScraperAPI `premium=true` paraméterével proxy hálózatot használtam, valamint **Tool-szintű belső újrapróbálkozási (Retry) logikát** írtam a kódba. Ha az API "Folyamatban" státuszt ad, a kód magától vár és újra megpróbálja (maximum 3-szor), így drasztikusan lecsökkentve az adathiányos válaszok számát.

## 3. Jövőbeli Fejlesztési Irányok (SFF Fókusz)

Bár az ágens jelenleg kiválóan képes TDP-t számolni és alapvető méreteket ellenőrizni, a Small Form Factor (SFF) PC-építés doménjében számos további kritikus peremfeltételt lehetne eszközként (Tool) implementálni a jövőben:

1.  **Fizikai Kompatibilitási Mátrix (Clearance Checker):**
    *   *Gépház dimenziók lekérése:* Egy új tool, ami kinyeri a gépházak űrtartalmát (liter), a maximális GPU hosszúságot és vastagságot (slottok száma), valamint a maximális CPU hűtő magasságot.
    *   *CPU Hűtő vs. RAM ütközés (Overhang):* SFF ITX alaplapoknál kritikus, hogy az alacsony profilú (low-profile) CPU hűtő rálóg-e a memóriamodulokra. Egy tool ellenőrizhetné a hűtő alatti szabad magasságot és a RAM magasságát.
2.  **Termális és Légáramlás (Airflow) Tanácsadó:**
    *   *Reddit Sentiment kibővítése:* A fórumokat ne csak általános hangulat (pozitív/negatív) alapján elemezzük, hanem fókuszáltan a hőmérsékleti problémákra. Például egy tool lekérdezhetné, hogy *"Mi az ajánlott undervolt beállítás a Ryzen 7 9800X3D-hez egy Dan A4-H2O házban?"*
    *   *Pozitív/Negatív légnyomás:* Az ágens kaphatna egy validációs logikát, amely figyelmezteti a felhasználót a szendvics-elrendezésű (kétkamrás) házaknál fellépő *Thermal Soaking* jelenségre és a megfelelő (kipufogó/beszívó) ventilátor orientációra.
3.  **Tápegység Formátum Validátor:**
    *   A TDP kiszámításán túl a rendszernek ellenőriznie kellene, hogy az adott gépház ATX, SFX-L vagy szigorúan csak SFX tápegységet fogad-e be, és felhívni a figyelmet az egyedi, rövidített kábelek (custom cables) szükségességére.